# ML-08 — Honest Model vs Week-4 Baseline

Train a transparent learned classifier for content-review prioritization and compare it with the Week-4 rule on the same grouped holdout. The decline label is retrospective and must never be used as a feature.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import recall_score, precision_score, f1_score, roc_auc_score, confusion_matrix
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

DATA_PATH = Path("data/raw/content_refresh_anonymized.csv")
if not DATA_PATH.exists() or DATA_PATH.stat().st_size == 0:
    raise FileNotFoundError("Starter dataset is not committed to the public repo. Run this notebook in the approved data environment or place the public-safe starter CSV at data/raw/content_refresh_anonymized.csv.")
df = pd.read_csv(DATA_PATH)
required = {"content_id","client_id","days_since_last_update","impressions_90d","trend_direction"}
missing = required - set(df.columns)
assert not missing, f"Missing required columns: {sorted(missing)}"
TARGET="is_declining_label"
if TARGET not in df.columns:
    df[TARGET]=(df["trend_direction"].astype(str).str.lower()=="down").astype(int)
print("Dataset:",df.shape,"| Positive rate:",round(df[TARGET].mean(),4))

## Split design

A client-grouped 80/20 holdout prevents the same client from appearing in both train and test. This is stricter than a random row split for cross-client transfer.

In [ ]:
SEED=42
gss=GroupShuffleSplit(n_splits=1,test_size=.20,random_state=SEED)
train_idx,test_idx=next(gss.split(df,groups=df["client_id"]))
train=df.iloc[train_idx].copy(); test=df.iloc[test_idx].copy()
assert set(train.client_id).isdisjoint(set(test.client_id))
print("Train rows:",len(train),"| Test rows:",len(test))
print("Train clients:",train.client_id.nunique(),"| Test clients:",test.client_id.nunique())

## Features and leakage control

Exclude the retrospective target, trend fields from which it is derived, IDs/grouping keys, and any product-generated baseline fields. Missing values are handled inside the training pipeline.

In [ ]:
FORBIDDEN={TARGET,"trend_direction","trend_pct","content_id","client_id","score","reason_code","action_label","freshness_bucket","volume_bucket"}
feature_cols=[c for c in df.columns if c not in FORBIDDEN]
X_train=train[feature_cols]; X_test=test[feature_cols]; y_train=train[TARGET]; y_test=test[TARGET]
num=X_train.select_dtypes(include=np.number).columns.tolist(); cat=[c for c in feature_cols if c not in num]
prep=ColumnTransformer([('num',Pipeline([('imp',SimpleImputer(strategy='median',add_indicator=True)),('scale',StandardScaler())]),num),('cat',Pipeline([('imp',SimpleImputer(strategy='most_frequent')),('ohe',OneHotEncoder(handle_unknown='ignore'))]),cat)])
model=Pipeline([('prep',prep),('clf',LogisticRegression(max_iter=1000,class_weight='balanced',random_state=SEED))])
model.fit(X_train,y_train)
model_prob=model.predict_proba(X_test)[:,1]; model_pred=(model_prob>=.5).astype(int)
assert not (set(FORBIDDEN) & set(feature_cols) - {TARGET})
print("Features:",len(feature_cols),"| Numeric:",len(num),"| Categorical:",len(cat))

## Model vs baseline

In [ ]:
def p_at_k(y_true,scores,k):
    y_true=np.asarray(y_true); scores=np.asarray(scores); k=min(k,len(y_true)); order=np.argsort(-scores)[:k]; return float(y_true[order].mean())
def row(name,pred,scores):
    return {'method':name,'recall':recall_score(y_test,pred,zero_division=0),'precision':precision_score(y_test,pred,zero_division=0),'f1':f1_score(y_test,pred,zero_division=0),'roc_auc':roc_auc_score(y_test,scores),'precision_at_20':p_at_k(y_test,scores,20),'precision_at_50':p_at_k(y_test,scores,50)}
baseline_flag=test["days_since_last_update"].fillna(0).ge(180)&test["impressions_90d"].fillna(0).ge(3000)
baseline_score=np.where(baseline_flag,test["impressions_90d"].fillna(0),0.0); baseline_pred=baseline_flag.astype(int)
comparison=pd.DataFrame([row("Week-4 rule baseline",baseline_pred,baseline_score),row("Logistic Regression",model_pred,model_prob)])
print(comparison.round(4).to_string(index=False))
print("Confusion matrix [TN FP; FN TP]:\n",confusion_matrix(y_test,model_pred))

## Error interpretation

Recall is primary because missing a declining page can leave a review opportunity undiscovered. Precision@20 and @50 assess whether the top of the ranked queue is useful. These metrics do not establish causality.

In [ ]:
coef=model.named_steps['clf'].coef_[0]
names=model.named_steps['prep'].get_feature_names_out()
coef_table=pd.DataFrame({'feature':names,'coefficient':coef}).assign(abs_coefficient=lambda d:d.coefficient.abs()).sort_values('abs_coefficient',ascending=False)
print(coef_table.head(15).round(3).to_string(index=False))
metrics={'seed':SEED,'split':'client_grouped_80_20','train_rows':len(train),'test_rows':len(test),'base_rate':float(df[TARGET].mean()),'comparison':comparison.to_dict(orient='records')}
out=Path('work/outputs'); out.mkdir(parents=True,exist_ok=True)
(out/'ml08_metrics.json').write_text(json.dumps(metrics,indent=2))
print('Saved:',out/'ml08_metrics.json')

## Honest conclusion

Use the learned score only if it improves the operational ranking on the same holdout without leakage. A higher score is not evidence that refreshing content causes recovery. The final capstone should prefer the model only when the measured comparison justifies it.